# 07. Final Aggregation

클러스터별 최적 모델을 적용하고 월별 총 수요를 집계하는 최종 노트북입니다.

| Cluster | 최종 모델 | Test MAPE |
| --- | --- | ---: |
| 1 | 날씨 + SARIMAX 25% + LightGBM 75% | 6.80% |
| 2 | 날씨 + SARIMAX + GridSearch | 5.74% |
| 3 | 날씨 + SARIMAX + GridSearch | 4.62% |
| 4 | 날씨 + 경제지수 + SARIMAX + GridSearch | 10.80% |

공통 전처리 로직은 `src/preprocessing.py`에 정리되어 있습니다.

In [50]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import prophet

In [51]:
final = pd.read_csv('../data/final.csv', encoding='utf-8-sig')
final

,판매월,제조업체,상품명,판매수량,대분류,중분류,한파,폭염,호우,생활물가지수,음식료품_계절조정지수
0,2021-01,88010455.0,(오뚜기)열라면종이(용기)105g,1.0,면류.라면류,라면류,22,0,0,101.20,105.5
1,2021-01,88030600.0,(백제물산)즉석쌀국수멸치맛92g,2.0,면류.라면류,면류,22,0,0,101.20,105.5
2,2021-01,88030600.0,(백제물산)즉석쌀국수멸치맛92g,1.0,면류.라면류,면류,22,0,0,101.20,105.5
3,2021-01,88010455.0,(오뚜기)열라면종이(용기)105g,2.0,면류.라면류,라면류,22,0,0,101.20,105.5
4,2021-01,88010455.0,(오뚜기)열라면종이(용기)105g,1.0,면류.라면류,라면류,22,0,0,101.20,105.5
...,...,...,...,...,...,...,...,...,...,...,...
80578,2024-02,NaN,NaN,NaN,NaN,NaN,2,0,12,116.32,91.0
80579,2024-03,NaN,NaN,NaN,NaN,NaN,5,0,6,116.62,94.5
80580,2024-04,NaN,NaN,NaN,NaN,NaN,0,0,0,116.59,94.0
80581,2024-05,NaN,NaN,NaN,NaN,NaN,0,0,21,116.53,94.6


In [52]:
final_1 = final.copy()
final_1.drop(columns=['상품명', '대분류', '중분류'], inplace=True)

In [53]:
# 2024년 데이터 삭제
final_1['판매월'] = pd.to_datetime(final_1['판매월'], format='%Y-%m')
final_1 = final_1[final_1['판매월'].dt.year != 2024]

In [54]:
# 제조업체별 판매수량 합계 계산 및 내림차순 정렬
total_sales_by_manufacturer = final_1.groupby('제조업체')['판매수량'].sum().reset_index().sort_values(by='판매수량', ascending=False).reset_index(drop=True)

# 목표 군집 수
n_clusters = 4

# 총 판매수량을 기준으로 균등하게 군집화
total_sales = total_sales_by_manufacturer['판매수량'].sum()  # 전체 판매수량 합계
target_sales_per_cluster = total_sales / n_clusters  # 각 군집이 목표로 하는 판매수량 합계

# 그리디 알고리즘을 사용하여 제조업체를 군집화
clusters = [[] for _ in range(n_clusters)]  # 각 군집을 담을 리스트
cluster_sales = [0] * n_clusters  # 각 군집의 현재 판매수량 합계
cluster_labels = []  # 제조업체에 대한 군집 레이블

for idx, row in total_sales_by_manufacturer.iterrows():
    # 가장 적은 판매수량을 가진 군집에 제조업체를 추가
    min_cluster_idx = np.argmin(cluster_sales)
    clusters[min_cluster_idx].append(row['제조업체'])
    cluster_sales[min_cluster_idx] += row['판매수량']
    cluster_labels.append(min_cluster_idx + 1)  # 군집 번호는 1부터 시작

# 결과를 데이터프레임으로 변환
total_sales_by_manufacturer['군집'] = cluster_labels

# 각 군집별 총 판매수량 출력
for i in range(n_clusters):
    cluster_total_sales = total_sales_by_manufacturer[total_sales_by_manufacturer['군집'] == i+1]['판매수량'].sum()
    print(f"Cluster {i+1} (총 판매수량: {cluster_total_sales})")

# 최종 결과 데이터프레임 출력
total_sales_by_manufacturer

Cluster 1 (총 판매수량: 106999.0)
Cluster 2 (총 판매수량: 69525.0)
Cluster 3 (총 판매수량: 39133.0)
Cluster 4 (총 판매수량: 39129.0)


,제조업체,판매수량,군집
0,88010430.0,106999.0,1
1,88010455.0,69525.0,2
2,88030600.0,26296.0,3
3,88010454.0,13272.0,4
4,88010731.0,9620.0,4
5,88011285.0,7499.0,4
6,88010453.0,6270.0,3
7,88010732.0,4617.0,4
8,88010438.0,3838.0,3
9,88096952.0,1796.0,4


In [55]:
final_1['cluster'] = final_1['제조업체'].map(dict(zip(total_sales_by_manufacturer['제조업체'], total_sales_by_manufacturer['군집'])))

In [56]:
final_1 = final_1.drop(columns=['제조업체'])

In [57]:
# 월별 cluster별 판매수량 합계 계산
final_1 = final_1.groupby(['판매월', 'cluster'], as_index=False).agg({
    '판매수량': 'sum',
    '한파': 'first',  # 동일한 값을 유지
    '폭염': 'first',  # 동일한 값을 유지
    '호우': 'first',   # 동일한 값을 유지
    '생활물가지수': 'first',  # 동일한 값을 유지
    '음식료품_계절조정지수': 'first'  # 동일한 값을 유지
})

## 2023-05 cluster4 이상치 대체
mean_cluster4 = final_1[(final_1['판매월'] < '2023-09-01') & (final_1['cluster'] == 4)]['판매수량'].mean()
final_1.loc[(final_1['판매월'] == '2023-05-01') & (final_1['cluster'] == 4), '판매수량'] = mean_cluster4

In [58]:
final_2 = final_1.copy()
final_2

,판매월,cluster,판매수량,한파,폭염,호우,생활물가지수,음식료품_계절조정지수
0,2021-01-01,1,3719.0,22,0,0,101.20,105.5
1,2021-01-01,2,2673.0,22,0,0,101.20,105.5
2,2021-01-01,3,1536.0,22,0,0,101.20,105.5
3,2021-01-01,4,880.0,22,0,0,101.20,105.5
4,2021-02-01,1,2654.0,12,0,0,102.11,97.1
...,...,...,...,...,...,...,...,...
139,2023-11-01,4,855.0,22,0,5,115.22,94.8
140,2023-12-01,1,3035.0,14,0,2,114.84,94.6
141,2023-12-01,2,1726.0,14,0,2,114.84,94.6
142,2023-12-01,3,881.0,14,0,2,114.84,94.6


### cluster별로 데이터 분리

In [59]:
cluster_1 = final_2[final_2['cluster'] == 1].reset_index(drop=True)
cluster_2 = final_2[final_2['cluster'] == 2].reset_index(drop=True)
cluster_3 = final_2[final_2['cluster'] == 3].reset_index(drop=True)
cluster_4 = final_2[final_2['cluster'] == 4].reset_index(drop=True)

In [60]:
cluster_1

,판매월,cluster,판매수량,한파,폭염,호우,생활물가지수,음식료품_계절조정지수
0,2021-01-01,1,3719.0,22,0,0,101.20,105.5
1,2021-02-01,1,2654.0,12,0,0,102.11,97.1
2,2021-03-01,1,4190.0,0,0,9,102.53,98.7
3,2021-04-01,1,3109.0,2,0,0,102.65,101.8
4,2021-05-01,1,2969.0,0,0,11,102.67,99.6
5,2021-06-01,1,2783.0,0,0,9,102.80,100.6
6,2021-07-01,1,3325.0,0,77,130,102.77,102.6
7,2021-08-01,1,2856.0,0,34,179,103.33,101.8
8,2021-09-01,1,2409.0,0,0,32,104.23,105.6
9,2021-10-01,1,3110.0,5,0,4,104.42,102.8


---

## cluster 1
### cluster + 날씨 / ensemble(lightgbm*0.75 + sarimax*0.25) / x

In [61]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_percentage_error

In [62]:
# 데이터 로드 및 날짜 형식 변환
cluster_1['판매월'] = pd.to_datetime(cluster_1['판매월'])

# 학습 데이터 (2021-01 ~ 2023-08)
train_data = cluster_1[(cluster_1['판매월'] >= '2021-01-01') & (cluster_1['판매월'] <= '2023-08-01')]

# 테스트 데이터 (2023-09 ~ 2023-12)
test_data = cluster_1[(cluster_1['판매월'] >= '2023-09-01') & (cluster_1['판매월'] <= '2023-12-01')]

# 외생변수 선택 (한파, 폭염, 호우)
exog_train = train_data[['한파', '폭염', '호우']]
exog_test = test_data[['한파', '폭염', '호우']]

# SARIMAX 모델 학습
sarimax_model = SARIMAX(train_data['판매수량'], exog=exog_train, order=(1,1,1), seasonal_order=(1,1,1,12))
sarimax_result = sarimax_model.fit()

# SARIMAX 예측
sarimax_pred = sarimax_result.predict(start=len(train_data), end=len(train_data) + len(test_data) - 1, exog=exog_test)

# LightGBM 모델 학습
lgbm_model = LGBMRegressor()
lgbm_model.fit(exog_train, train_data['판매수량'])

# LightGBM 예측
lgbm_pred = lgbm_model.predict(exog_test)

# 앙상블 예측 (SARIMAX 25%, LGBM 75%)
ensemble_pred = 0.25 * sarimax_pred + 0.75 * lgbm_pred

# 실제 판매수량
actual = test_data['판매수량'].values

# MAPE 계산 (백분율로 변환)
sarimax_mape = mean_absolute_percentage_error(actual, sarimax_pred) * 100
lgbm_mape = mean_absolute_percentage_error(actual, lgbm_pred) * 100
ensemble_mape = mean_absolute_percentage_error(actual, ensemble_pred) * 100

RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =            8     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  4.58425D+00    |proj g|=  1.38431D-01

At iterate    5    f=  4.51845D+00    |proj g|=  1.60672D-02

At iterate   10    f=  4.51709D+00    |proj g|=  2.73503D-03


 This problem is unconstrained.



At iterate   15    f=  4.51691D+00    |proj g|=  2.15245D-03

At iterate   20    f=  4.51661D+00    |proj g|=  3.24298D-03

At iterate   25    f=  4.51169D+00    |proj g|=  2.37341D-02

At iterate   30    f=  4.50762D+00    |proj g|=  5.77541D-03

At iterate   35    f=  4.50692D+00    |proj g|=  7.67524D-03

At iterate   40    f=  4.49791D+00    |proj g|=  9.74395D-03

At iterate   45    f=  4.49731D+00    |proj g|=  3.40920D-04

At iterate   50    f=  4.49716D+00    |proj g|=  1.18596D-03

           * * *

Tit   = total number of iterations
Tnf   = total number of function evaluations
Tnint = total number of segments explored during Cauchy searches
Skip  = number of BFGS updates skipped
Nact  = number of active bounds at final generalized Cauchy point
Projg = norm of the final projected gradient
F     = final function value

           * * *

   N    Tit     Tnf  Tnint  Skip  Nact     Projg        F
    8     50     55      1     0     0   1.186D-03   4.497D+00
  F =   4.49715931163

In [63]:
# 결과 출력 데이터프레임 생성
results = pd.DataFrame({
    '판매월': test_data['판매월'],
    '실제 판매수량': actual,
    'SARIMAX 예측': sarimax_pred,
    'LGBM 예측': lgbm_pred,
    '앙상블 예측': ensemble_pred
})

results

,판매월,실제 판매수량,SARIMAX 예측,LGBM 예측,앙상블 예측
32,2023-09-01,2752.0,2198.145757,2958.59375,2768.481752
33,2023-10-01,3205.0,2254.804005,2958.59375,2782.646314
34,2023-11-01,3332.0,2671.505322,2958.59375,2886.821643
35,2023-12-01,3035.0,3258.844418,2958.59375,3033.656417


In [64]:
print(f'SARIMAX MAPE: {sarimax_mape:.2f}%')
print(f'LGBM MAPE: {lgbm_mape:.2f}%')
print(f'앙상블 MAPE: {ensemble_mape:.2f}%')

SARIMAX MAPE: 19.24%
LGBM MAPE: 7.23%
앙상블 MAPE: 6.80%


---

## cluster 2
### cluster + 날씨 / sarimax / GridSearchCV

In [65]:
import pandas as pd
import numpy as np
from itertools import product
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings("ignore")  # 경고문 무시

In [66]:
# SARIMA 파라미터 조합 설정
p = d = q = range(0, 2)
seasonal_p = seasonal_d = seasonal_q = range(0, 2)
seasonal_period = [12]  # 월 단위 시즌(12개월)
param_combinations = list(product(p, d, q))
seasonal_combinations = list(product(seasonal_p, seasonal_d, seasonal_q, seasonal_period))

# SARIMAX 모델 학습 및 예측 함수
def train_and_predict_sarimax(cluster_data):
    global final_results

    # 학습 및 테스트 데이터 준비
    train_data = cluster_data[(cluster_data['판매월'] >= '2021-01-01') & (cluster_data['판매월'] <= '2023-08-01')]['판매수량']
    test_data = cluster_data[(cluster_data['판매월'] >= '2023-09-01') & (cluster_data['판매월'] <= '2023-12-01')]['판매수량']

    # 외생변수 준비 (학습 및 테스트)
    train_exog = cluster_data[(cluster_data['판매월'] >= '2021-01-01') & (cluster_data['판매월'] <= '2023-08-01')][exog_vars]
    test_exog = cluster_data[(cluster_data['판매월'] >= '2023-09-01') & (cluster_data['판매월'] <= '2023-12-01')][exog_vars]

    # 최적의 SARIMAX 파라미터 찾기 (grid search)
    best_mape = float('inf')
    best_params = None
    best_seasonal_params = None
    best_model = None
    for params in param_combinations:
        for seasonal_params in seasonal_combinations:
            try:
                # SARIMAX 모델 학습
                sarimax_model = SARIMAX(train_data, exog=train_exog, order=params, seasonal_order=seasonal_params)
                sarimax_result = sarimax_model.fit(disp=False)

                # 2023년 9월부터 12월까지 예측 (외생변수 반영)
                forecast = sarimax_result.predict(start=len(train_data), end=len(train_data) + len(test_data) - 1, exog=test_exog)

                # MAPE 계산
                mape = mean_absolute_percentage_error(test_data, forecast)

                # 최적의 모델 찾기
                if mape < best_mape:
                    best_mape = mape
                    best_params = params
                    best_seasonal_params = seasonal_params
                    best_model = sarimax_result

            except Exception as e:
                # 모델이 실패하는 경우를 무시하고 다음 파라미터로 진행
                continue

    # 최적 모델로 예측
    forecast = best_model.predict(start=len(train_data), end=len(train_data) + len(test_data) - 1, exog=test_exog)

    # 실제값과 예측값 비교 데이터프레임 생성
    comparison_df = pd.DataFrame({
        '판매월': cluster_data[(cluster_data['판매월'] >= '2023-09-01') & (cluster_data['판매월'] <= '2023-12-01')]['판매월'],
        '실제수량': test_data.values,
        '예측수량': forecast.values,
    })

    # 결과를 최종 데이터프레임에 추가
    final_results = pd.concat([final_results, comparison_df])

    return best_mape * 100, best_params, best_seasonal_params  # MAPE를 백분율로 변환

In [67]:
# 외생변수 선택
exog_vars = ['한파', '폭염', '호우']

In [68]:
# 결과 저장을 위한 빈 데이터프레임
final_results = pd.DataFrame()

# SARIMAX 모델 적용
mape, best_params, best_seasonal_params = train_and_predict_sarimax(cluster_2)

# 최종 MAPE 계산 (전체 데이터에 대한 MAPE)
total_mape = mean_absolute_percentage_error(final_results['실제수량'], final_results['예측수량']) * 100

In [69]:
# 결과 출력
print(f"=== 최적의 파라미터 ===")
print(f"SARIMA order: {best_params}")
print(f"Seasonal order: {best_seasonal_params}")
print(f"\n=== 최종 MAPE ===")
print(f"Total MAPE: {total_mape:.2f}%")

=== 최적의 파라미터 ===
SARIMA order: (0, 1, 1)
Seasonal order: (0, 0, 0, 12)

=== 최종 MAPE ===
Total MAPE: 5.74%


In [70]:
final_results

,판매월,실제수량,예측수량
32,2023-09-01,1704.0,1702.949743
33,2023-10-01,2032.0,1792.112600
34,2023-11-01,1968.0,1917.154273
35,2023-12-01,1726.0,1872.601147


---

## cluster 3
### cluster + 날씨 / sarimax / GridSearchCV

In [71]:
# 외생변수 선택
exog_vars = ['한파', '폭염', '호우']

In [72]:
final_results = pd.DataFrame()

# SARIMAX 모델 적용
mape, best_params, best_seasonal_params = train_and_predict_sarimax(cluster_3)

# 최종 MAPE 계산 (전체 데이터에 대한 MAPE)
total_mape = mean_absolute_percentage_error(final_results['실제수량'], final_results['예측수량']) * 100

In [73]:
# 결과 출력
print(f"=== 최적의 파라미터 ===")
print(f"SARIMA order: {best_params}")
print(f"Seasonal order: {best_seasonal_params}")
print(f"\n=== 최종 MAPE ===")
print(f"Total MAPE: {total_mape:.2f}%")

=== 최적의 파라미터 ===
SARIMA order: (1, 1, 1)
Seasonal order: (0, 1, 1, 12)

=== 최종 MAPE ===
Total MAPE: 4.62%


In [74]:
final_results

,판매월,실제수량,예측수량
32,2023-09-01,973.0,980.158814
33,2023-10-01,924.0,954.407047
34,2023-11-01,640.0,631.648557
35,2023-12-01,881.0,996.826406


---

## cluster 4
### cluster + 날씨 + 경제지수  /  sarimax  / gridsearchCV

In [75]:
# 외생변수 선택
exog_vars = ['한파', '폭염', '호우', '생활물가지수', '음식료품_계절조정지수']

In [76]:
final_results = pd.DataFrame()

# SARIMAX 모델 적용
mape, best_params, best_seasonal_params = train_and_predict_sarimax(cluster_4)

# 최종 MAPE 계산 (전체 데이터에 대한 MAPE)
total_mape = mean_absolute_percentage_error(final_results['실제수량'], final_results['예측수량']) * 100

In [77]:
# 결과 출력
print(f"=== 최적의 파라미터 ===")
print(f"SARIMA order: {best_params}")
print(f"Seasonal order: {best_seasonal_params}")
print(f"\n=== 최종 MAPE ===")
print(f"Total MAPE: {total_mape:.2f}%")

=== 최적의 파라미터 ===
SARIMA order: (0, 1, 1)
Seasonal order: (0, 0, 0, 12)

=== 최종 MAPE ===
Total MAPE: 10.80%


In [78]:
final_results

,판매월,실제수량,예측수량
32,2023-09-01,1169.0,1121.136847
33,2023-10-01,1361.0,1164.966701
34,2023-11-01,855.0,912.295330
35,2023-12-01,1206.0,988.907371


---

In [82]:
final_2

,판매월,cluster,판매수량,한파,폭염,호우,생활물가지수,음식료품_계절조정지수
0,2021-01-01,1,3719.0,22,0,0,101.20,105.5
1,2021-01-01,2,2673.0,22,0,0,101.20,105.5
2,2021-01-01,3,1536.0,22,0,0,101.20,105.5
3,2021-01-01,4,880.0,22,0,0,101.20,105.5
4,2021-02-01,1,2654.0,12,0,0,102.11,97.1
...,...,...,...,...,...,...,...,...
139,2023-11-01,4,855.0,22,0,5,115.22,94.8
140,2023-12-01,1,3035.0,14,0,2,114.84,94.6
141,2023-12-01,2,1726.0,14,0,2,114.84,94.6
142,2023-12-01,3,881.0,14,0,2,114.84,94.6


In [87]:
# '판매월'을 datetime 형식으로 변환
final_2['판매월'] = pd.to_datetime(final_2['판매월'])

# 2023년 9월부터 12월까지의 데이터 필터링 및 판매수량 총 집계
monthly_sales = final_2[(final_2['판매월'] >= '2023-09-01') & (final_2['판매월'] <= '2023-12-31')]
monthly_sales_agg = monthly_sales.groupby(monthly_sales['판매월'].dt.to_period('M'))['판매수량'].sum().reset_index()

# 주어진 예측판매수량 값들
predicted_sales = [6572.727156, 6694.132662, 6347.919803, 6891.991341]

# 예측판매수량 칼럼 추가
monthly_sales_agg['예측판매수량'] = predicted_sales

# 실제 판매수량과 예측 판매수량의 차이 계산하여 새로운 칼럼 추가
monthly_sales_agg['차이'] = abs(monthly_sales_agg['판매수량'] - monthly_sales_agg['예측판매수량'])

# MAPE 계산 함수
def calculate_mape(actual, predicted):
    return abs((actual - predicted) / actual) * 100

# 각 월별 MAPE 계산 및 추가
monthly_sales_agg['mape'] = monthly_sales_agg.apply(
    lambda row: calculate_mape(row['판매수량'], row['예측판매수량']), axis=1
)

# 결과 출력
monthly_sales_agg

,판매월,판매수량,예측판매수량,차이,mape
0,2023-09,6598.0,6572.727156,25.272844,0.383038
1,2023-10,7522.0,6694.132662,827.867338,11.005947
2,2023-11,6795.0,6347.919803,447.080197,6.579547
3,2023-12,6848.0,6891.991341,43.991341,0.642397


In [88]:
monthly_sales_agg['차이'].sum()

1344.2117200000002